# M9 — Machine Learning Feasibility and Leakage Audit

## Objective

Assess whether UNSW-NB15 can support a reproducible
binary intrusion detection classification workflow.

## Scope

- Dataset: UNSW-NB15
- Target: label
- Task: Binary classification
- Candidate models: Logistic Regression and Random Forest

## Key Checks

1. Feature schema and data types
2. Target distribution
3. Target-derived columns
4. Duplicate overlap between partitions
5. Preprocessing requirements

The official test partition will remain reserved
for final model evaluation.

In [1]:
import pandas as pd
import numpy as np

from pathlib import Path

current_dir = Path.cwd()

project_dir = (
    current_dir.parent
    if current_dir.name == "notebooks"
    else current_dir
)

data_dir = project_dir / "data" / "raw" / "unsw_nb15"

train_path = data_dir / "UNSW_NB15_training-set.csv"
test_path = data_dir / "UNSW_NB15_testing-set.csv"

assert train_path.exists()
assert test_path.exists()

print("SUCCESS: UNSW-NB15 files located!")

SUCCESS: UNSW-NB15 files located!


In [2]:
train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)

print("Training shape:", train_df.shape)
print("Testing shape:", test_df.shape)

assert train_df.shape == (175341, 45)
assert test_df.shape == (82332, 45)

print("\nSUCCESS: Dataset dimensions verified!")

Training shape: (175341, 45)
Testing shape: (82332, 45)

SUCCESS: Dataset dimensions verified!


In [3]:
schema_df = pd.DataFrame({
    "Column": train_df.columns,
    "Data Type": train_df.dtypes.astype(str).values,
    "Unique Values": train_df.nunique().values,
    "Missing Values": train_df.isna().sum().values
})

display(schema_df)

,Column,Data Type,Unique Values,Missing Values
0,id,int64,175341,0
1,dur,float64,74039,0
2,proto,str,133,0
3,service,str,13,0
4,state,str,9,0
5,spkts,int64,480,0
6,dpkts,int64,443,0
7,sbytes,int64,7214,0
8,dbytes,int64,6660,0
9,rate,float64,76991,0


In [4]:
target_column = "label"

excluded_columns = [
    "id",
    "attack_cat",
    "label"
]

feature_columns = [
    column
    for column in train_df.columns
    if column not in excluded_columns
]

X_train_full = train_df[feature_columns].copy()
y_train_full = train_df[target_column].copy()

X_test_final = test_df[feature_columns].copy()
y_test_final = test_df[target_column].copy()

print("Number of ML features:", len(feature_columns))

print("Training features:", X_train_full.shape)
print("Testing features:", X_test_final.shape)

print("\nCategorical columns:")
print(
    X_train_full.select_dtypes(
        include=["object", "category", "str"]
    ).columns.tolist()
)

print("\nTarget distribution:")
print(y_train_full.value_counts())

assert len(feature_columns) == 42
assert "label" not in X_train_full.columns
assert "attack_cat" not in X_train_full.columns
assert "id" not in X_train_full.columns

print("\nSUCCESS: ML features and target separated!")

Number of ML features: 42
Training features: (175341, 42)
Testing features: (82332, 42)

Categorical columns:
['proto', 'service', 'state']

Target distribution:
label
1    119341
0     56000
Name: count, dtype: int64

SUCCESS: ML features and target separated!


In [5]:
assert X_train_full.columns.tolist() == (
    X_test_final.columns.tolist()
)

assert not X_train_full.isna().any().any()
assert not X_test_final.isna().any().any()

assert set(y_train_full.unique()) == {0, 1}
assert set(y_test_final.unique()) == {0, 1}

print("SUCCESS: Train-test feature schema verified!")

SUCCESS: Train-test feature schema verified!


In [6]:
ml_reports_dir = project_dir / "reports" / "ml"

ml_reports_dir.mkdir(
    parents=True,
    exist_ok=True
)

schema_output_path = (
    ml_reports_dir / "unsw_ml_feature_schema.csv"
)

schema_df.to_csv(
    schema_output_path,
    index=False
)

print("Saved:", schema_output_path)

Saved: C:\Users\fatim\Documents\network-intrusion-eda\reports\ml\unsw_ml_feature_schema.csv


## M9.2 — Duplicate Overlap and Leakage Audit

### Objective

Investigate duplicate feature patterns within and across
the official UNSW-NB15 training and testing partitions.

### Questions

1. How many unique feature patterns exist?
2. How many testing records match training feature patterns?
3. Are overlapping patterns associated with conflicting labels?
4. What are the implications for model evaluation?

### Important

The official testing partition is used only for
dataset-integrity auditing at this stage.

It will not be used for model training, hyperparameter
selection, or decision-threshold selection.

In [7]:
train_unique = X_train_full.drop_duplicates()

test_unique = X_test_final.drop_duplicates()

train_unique_count = len(train_unique)
test_unique_count = len(test_unique)

print("Training records:", len(X_train_full))
print("Unique training feature patterns:", train_unique_count)

print()

print("Testing records:", len(X_test_final))
print("Unique testing feature patterns:", test_unique_count)

print()

print(
    "Training feature duplicate percentage:",
    round(
        (1 - train_unique_count / len(X_train_full)) * 100,
        2
    )
)

print(
    "Testing feature duplicate percentage:",
    round(
        (1 - test_unique_count / len(X_test_final)) * 100,
        2
    )
)

Training records: 175341
Unique training feature patterns: 101040

Testing records: 82332
Unique testing feature patterns: 53946

Training feature duplicate percentage: 42.38
Testing feature duplicate percentage: 34.48


In [8]:
train_hashes = pd.util.hash_pandas_object(
    X_train_full,
    index=False
)

test_hashes = pd.util.hash_pandas_object(
    X_test_final,
    index=False
)

train_hash_set = set(train_hashes)

test_overlap_mask = test_hashes.isin(train_hash_set)

overlap_count = int(test_overlap_mask.sum())

overlap_percentage = (
    overlap_count / len(X_test_final)
) * 100

print("Testing records:", len(X_test_final))
print("Testing records matching training feature hashes:", overlap_count)

print(
    "Testing overlap percentage:",
    round(overlap_percentage, 2)
)

print(
    "Unique overlapping feature hashes:",
    test_hashes[test_overlap_mask].nunique()
)

Testing records: 82332
Testing records matching training feature hashes: 8541
Testing overlap percentage: 10.37
Unique overlapping feature hashes: 1302


In [9]:
# Create hash-label tables

train_hash_label_df = pd.DataFrame({
    "feature_hash": train_hashes.to_numpy(),
    "label": y_train_full.to_numpy()
})

test_hash_label_df = pd.DataFrame({
    "feature_hash": test_hashes.to_numpy(),
    "label": y_test_final.to_numpy()
})


# Get all labels associated with each training hash

train_label_sets = (
    train_hash_label_df
    .groupby("feature_hash")["label"]
    .agg(lambda values: set(values))
)


# Select testing records whose hashes appear in training

test_overlap_labels = test_hash_label_df.loc[
    test_overlap_mask.to_numpy()
].copy()


# Check conflicts without DataFrame.apply(axis=1)

label_conflicts = []

for feature_hash, test_label in zip(
    test_overlap_labels["feature_hash"],
    test_overlap_labels["label"]
):

    known_labels = train_label_sets.loc[feature_hash]

    label_conflicts.append(
        test_label not in known_labels
    )


test_overlap_labels["label_conflict"] = label_conflicts


# Calculate results

conflict_count = int(
    test_overlap_labels["label_conflict"].sum()
)

print(
    "Overlapping test records:",
    len(test_overlap_labels)
)

print(
    "Overlapping test records with unseen label:",
    conflict_count
)

conflict_percentage = (
    conflict_count / len(test_overlap_labels) * 100
    if len(test_overlap_labels) > 0
    else 0
)

print(
    "Conflict percentage among overlapping test records:",
    round(conflict_percentage, 2)
)

print("\nSUCCESS: Overlapping label audit completed!")

Overlapping test records: 8541
Overlapping test records with unseen label: 296
Conflict percentage among overlapping test records: 3.47

SUCCESS: Overlapping label audit completed!


In [10]:
print(
    "Training hash dtype:",
    train_hash_label_df["feature_hash"].dtype
)

print(
    "Testing hash dtype:",
    test_hash_label_df["feature_hash"].dtype
)

assert str(
    train_hash_label_df["feature_hash"].dtype
) == "uint64"

assert str(
    test_hash_label_df["feature_hash"].dtype
) == "uint64"

assert conflict_count <= overlap_count

print("\nSUCCESS: Hash precision and label checks passed!")

Training hash dtype: uint64
Testing hash dtype: uint64

SUCCESS: Hash precision and label checks passed!


In [11]:
train_label_counts = (
    train_hash_label_df
    .groupby("feature_hash")["label"]
    .nunique()
)

ambiguous_train_hashes = train_label_counts[
    train_label_counts > 1
]

print(
    "Training feature hashes with multiple labels:",
    len(ambiguous_train_hashes)
)

print(
    "Training records belonging to ambiguous feature hashes:",
    int(
        train_hashes.isin(
            ambiguous_train_hashes.index
        ).sum()
    )
)

Training feature hashes with multiple labels: 229
Training records belonging to ambiguous feature hashes: 940


In [12]:
overlap_audit = pd.DataFrame([
    {
        "Metric": "Training records",
        "Value": len(X_train_full)
    },
    {
        "Metric": "Testing records",
        "Value": len(X_test_final)
    },
    {
        "Metric": "Unique training feature patterns",
        "Value": train_unique_count
    },
    {
        "Metric": "Unique testing feature patterns",
        "Value": test_unique_count
    },
    {
        "Metric": "Testing records matching training hashes",
        "Value": overlap_count
    },
    {
        "Metric": "Testing overlap percentage",
        "Value": round(overlap_percentage, 2)
    },
    {
        "Metric": "Overlapping test records with unseen label",
        "Value": conflict_count
    },
    {
        "Metric": "Ambiguous training feature hashes",
        "Value": len(ambiguous_train_hashes)
    }
])

display(overlap_audit)

output_path = (
    ml_reports_dir / "unsw_train_test_overlap_audit.csv"
)

overlap_audit.to_csv(
    output_path,
    index=False
)

print("\nSaved:", output_path)

,Metric,Value
0,Training records,175341.00
1,Testing records,82332.00
2,Unique training feature patterns,101040.00
3,Unique testing feature patterns,53946.00
4,Testing records matching training hashes,8541.00
5,Testing overlap percentage,10.37
6,Overlapping test records with unseen label,296.00
7,Ambiguous training feature hashes,229.00



Saved: C:\Users\fatim\Documents\network-intrusion-eda\reports\ml\unsw_train_test_overlap_audit.csv


In [13]:
assert train_unique_count <= len(X_train_full)

assert test_unique_count <= len(X_test_final)

assert 0 <= overlap_count <= len(X_test_final)

assert 0 <= conflict_count <= overlap_count

assert output_path.exists()

print("SUCCESS: M9.2 overlap audit completed!")

SUCCESS: M9.2 overlap audit completed!


### M9.2 — Exact-Match Verification

Hash-based overlap detection is followed by direct
comparison of the original feature values.

This verification distinguishes genuine matching
feature vectors from possible hash collisions.

The verified results will guide supplementary
model evaluation.

In [14]:
# Identify candidate hashes shared by both partitions

shared_hashes = pd.Index(
    train_hashes.unique()
).intersection(
    pd.Index(test_hashes.unique())
)

# Retain original feature values and their hashes

train_candidates = X_train_full.loc[
    train_hashes.isin(shared_hashes)
].copy()

test_candidates = X_test_final.loc[
    test_hashes.isin(shared_hashes)
].copy()

train_candidates["feature_hash"] = (
    train_hashes.loc[train_candidates.index].to_numpy()
)

test_candidates["feature_hash"] = (
    test_hashes.loc[test_candidates.index].to_numpy()
)

# Remove repeated feature vectors to reduce memory use

train_candidate_unique = train_candidates.drop_duplicates()

test_candidate_unique = test_candidates.drop_duplicates()

# Compare all 42 original features directly

exact_matches = test_candidate_unique.merge(
    train_candidate_unique,
    on=feature_columns,
    how="inner",
    suffixes=("_test", "_train")
)

# Both hash columns should agree for identical features

assert (
    exact_matches["feature_hash_test"]
    == exact_matches["feature_hash_train"]
).all()

verified_hashes = set(
    exact_matches["feature_hash_test"]
)

verified_overlap_mask = test_hashes.isin(
    verified_hashes
)

verified_overlap_count = int(
    verified_overlap_mask.sum()
)

print(
    "Verified overlapping test records:",
    verified_overlap_count
)

print(
    "Verified overlap percentage:",
    round(
        verified_overlap_count / len(X_test_final) * 100,
        2
    )
)

print(
    "Verified overlapping feature hashes:",
    len(verified_hashes)
)

print(
    "Non-overlapping test records:",
    len(X_test_final) - verified_overlap_count
)

print("\nSUCCESS: Exact-match verification completed!")

Verified overlapping test records: 8541
Verified overlap percentage: 10.37
Verified overlapping feature hashes: 1302
Non-overlapping test records: 73791

SUCCESS: Exact-match verification completed!


In [15]:
assert verified_overlap_count <= overlap_count

assert verified_overlap_count >= 0

assert len(verified_hashes) <= len(shared_hashes)

print("SUCCESS: Verified overlap counts are consistent!")

SUCCESS: Verified overlap counts are consistent!


## M9.3 — Safe ML Preprocessing Design

### Objective

Define reproducible preprocessing for binary classification
using UNSW-NB15.

### Design Principles

1. Fit preprocessing only on training data.
2. Keep the official test partition untouched during model selection.
3. Encode categorical features safely.
4. Handle missing and unseen values.
5. Use model-appropriate numerical transformations.
6. Prevent duplicate feature patterns from crossing
   the development training/validation boundary.

In [16]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import (
    OneHotEncoder,
    StandardScaler
)
from sklearn.impute import SimpleImputer
from sklearn.model_selection import GroupShuffleSplit

print("SUCCESS: ML preprocessing libraries imported!")

SUCCESS: ML preprocessing libraries imported!


In [17]:
categorical_features = [
    "proto",
    "service",
    "state"
]

numerical_features = [
    column
    for column in feature_columns
    if column not in categorical_features
]

print("Numerical features:", len(numerical_features))
print("Categorical features:", len(categorical_features))

print("\nCategorical columns:")
print(categorical_features)

assert len(numerical_features) == 39
assert len(categorical_features) == 3

assert set(numerical_features).isdisjoint(
    categorical_features
)

assert set(
    numerical_features + categorical_features
) == set(feature_columns)

print("\nSUCCESS: Feature groups verified!")

Numerical features: 39
Categorical features: 3

Categorical columns:
['proto', 'service', 'state']

SUCCESS: Feature groups verified!


In [18]:
numeric_pipeline_lr = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="median")
        ),
        (
            "scaler",
            StandardScaler()
        )
    ]
)

categorical_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="most_frequent")
        ),
        (
            "encoder",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=True
            )
        )
    ]
)

preprocessor_lr = ColumnTransformer(
    transformers=[
        (
            "numeric",
            numeric_pipeline_lr,
            numerical_features
        ),
        (
            "categorical",
            categorical_pipeline,
            categorical_features
        )
    ],
    sparse_threshold=0.3
)

print("SUCCESS: Logistic Regression preprocessing defined!")

SUCCESS: Logistic Regression preprocessing defined!


In [19]:
numeric_pipeline_rf = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="median")
        )
    ]
)

preprocessor_rf = ColumnTransformer(
    transformers=[
        (
            "numeric",
            numeric_pipeline_rf,
            numerical_features
        ),
        (
            "categorical",
            categorical_pipeline,
            categorical_features
        )
    ],
    sparse_threshold=0.3
)

print("SUCCESS: Random Forest preprocessing defined!")

SUCCESS: Random Forest preprocessing defined!


In [20]:
groups = train_hashes.to_numpy()

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_indices, val_indices = next(
    splitter.split(
        X_train_full,
        y_train_full,
        groups=groups
    )
)

X_dev_train = X_train_full.iloc[
    train_indices
].copy()

X_dev_val = X_train_full.iloc[
    val_indices
].copy()

y_dev_train = y_train_full.iloc[
    train_indices
].copy()

y_dev_val = y_train_full.iloc[
    val_indices
].copy()

print("Development training:", X_dev_train.shape)
print("Validation:", X_dev_val.shape)

print("\nTraining class proportions:")
print(y_dev_train.value_counts(normalize=True))

print("\nValidation class proportions:")
print(y_dev_val.value_counts(normalize=True))

Development training: (140145, 42)
Validation: (35196, 42)

Training class proportions:
label
1    0.679782
0    0.320218
Name: proportion, dtype: float64

Validation class proportions:
label
1    0.68397
0    0.31603
Name: proportion, dtype: float64


In [21]:
train_group_hashes = set(
    groups[train_indices]
)

val_group_hashes = set(
    groups[val_indices]
)

shared_groups = (
    train_group_hashes & val_group_hashes
)

print(
    "Shared feature hashes between development partitions:",
    len(shared_groups)
)

assert len(shared_groups) == 0

assert len(X_dev_train) + len(X_dev_val) == len(
    X_train_full
)

assert set(y_dev_train.unique()) == {0, 1}
assert set(y_dev_val.unique()) == {0, 1}

print("\nSUCCESS: Group-aware validation split verified!")

Shared feature hashes between development partitions: 0

SUCCESS: Group-aware validation split verified!


In [22]:
split_dir = project_dir / "reports" / "ml"

split_dir.mkdir(
    parents=True,
    exist_ok=True
)

split_assignments = pd.DataFrame({
    "original_row_index": np.concatenate([
        train_indices,
        val_indices
    ]),
    "split": (
        ["train"] * len(train_indices)
        + ["validation"] * len(val_indices)
    )
})

split_assignments = split_assignments.sort_values(
    "original_row_index"
).reset_index(drop=True)

split_path = (
    split_dir / "unsw_development_split.csv"
)

split_assignments.to_csv(
    split_path,
    index=False
)

print("Saved:", split_path)

assert len(split_assignments) == len(X_train_full)

print("SUCCESS: Development split saved!")

Saved: C:\Users\fatim\Documents\network-intrusion-eda\reports\ml\unsw_development_split.csv
SUCCESS: Development split saved!


## M10.1 — Logistic Regression Baseline

### Objective

Train the first binary intrusion detection classifier using
the UNSW-NB15 development training partition.

### Methodology

- Model: Logistic Regression
- Target: Normal (0) / Attack (1)
- Training: Group-aware development training partition
- Evaluation: Development validation partition
- Preprocessing: Imputation, scaling, one-hot encoding
- Official test: Not used

### Evaluation Metrics

Accuracy, precision, recall, F1-score, confusion matrix,
and average precision (AP).

In [23]:
from sklearn.base import clone
from sklearn.linear_model import LogisticRegression

In [24]:
lr_pipeline = Pipeline(
    steps=[
        (
            "preprocessing",
            clone(preprocessor_lr)
        ),
        (
            "classifier",
            LogisticRegression(
                solver="liblinear",
                max_iter=500,
                random_state=42
            )
        )
    ]
)

print(lr_pipeline)

print("\nSUCCESS: Logistic Regression pipeline created!")

Pipeline(steps=[('preprocessing',
                 ColumnTransformer(transformers=[('numeric',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['dur', 'spkts', 'dpkts',
                                                   'sbytes', 'dbytes', 'rate',
                                                   'sttl', 'dttl', 'sload',
                                                   'dload', 'sloss', 'dloss',
                                                   'sinpkt', 'dinpkt', 'sjit',
                                                   'djit', 'swin', 'stcpb',
                                                   'dtcpb', 'dwin', 'tcprtt',
                  

In [25]:
import time

print("Starting Logistic Regression training...")

start_time = time.time()

lr_pipeline.fit(
    X_dev_train,
    y_dev_train
)

lr_training_seconds = time.time() - start_time

print("\nTraining completed!")

print(
    "Training time:",
    round(lr_training_seconds, 2),
    "seconds"
)

print("\nSUCCESS: Logistic Regression trained!")

Starting Logistic Regression training...

Training completed!
Training time: 17.86 seconds

SUCCESS: Logistic Regression trained!


In [26]:
lr_val_predictions = lr_pipeline.predict(
    X_dev_val
)

lr_val_probabilities = (
    lr_pipeline.predict_proba(X_dev_val)[:, 1]
)

print("Validation records:", len(X_dev_val))

print("\nPredicted classes:")
print(
    pd.Series(lr_val_predictions).value_counts()
)

print("\nSUCCESS: Validation predictions generated!")

Validation records: 35196

Predicted classes:
1    25703
0     9493
Name: count, dtype: int64

SUCCESS: Validation predictions generated!


In [27]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    average_precision_score,
    confusion_matrix,
    classification_report
)

print("SUCCESS: All evaluation metrics imported!")

SUCCESS: All evaluation metrics imported!


In [28]:
lr_metrics = {
    "Model": "Logistic Regression",
    "Accuracy": accuracy_score(
        y_dev_val,
        lr_val_predictions
    ),
    "Precision": precision_score(
        y_dev_val,
        lr_val_predictions,
        zero_division=0
    ),
    "Recall": recall_score(
        y_dev_val,
        lr_val_predictions,
        zero_division=0
    ),
    "F1": f1_score(
        y_dev_val,
        lr_val_predictions,
        zero_division=0
    ),
    "Average Precision": average_precision_score(
        y_dev_val,
        lr_val_probabilities
    ),
    "Training Seconds": lr_training_seconds
}

lr_metrics_df = pd.DataFrame([lr_metrics])

display(lr_metrics_df.round(4))

print("\nSUCCESS: Baseline metrics calculated!")

,Model,Accuracy,Precision,Recall,F1,Average Precision,Training Seconds
0,Logistic Regression,0.9358,0.9244,0.987,0.9546,0.992,17.8637



SUCCESS: Baseline metrics calculated!


In [29]:
lr_cm = confusion_matrix(
    y_dev_val,
    lr_val_predictions,
    labels=[0, 1]
)

lr_cm_df = pd.DataFrame(
    lr_cm,
    index=["Actual Normal", "Actual Attack"],
    columns=["Predicted Normal", "Predicted Attack"]
)

display(lr_cm_df)

print("\nClassification Report:")

print(
    classification_report(
        y_dev_val,
        lr_val_predictions,
        target_names=["Normal", "Attack"],
        zero_division=0
    )
)

,Predicted Normal,Predicted Attack
Actual Normal,9179,1944
Actual Attack,314,23759



Classification Report:
              precision    recall  f1-score   support

      Normal       0.97      0.83      0.89     11123
      Attack       0.92      0.99      0.95     24073

    accuracy                           0.94     35196
   macro avg       0.95      0.91      0.92     35196
weighted avg       0.94      0.94      0.93     35196



In [30]:
baseline_metrics_path = (
    ml_reports_dir / "unsw_lr_baseline_metrics.csv"
)

lr_metrics_df.to_csv(
    baseline_metrics_path,
    index=False
)

assert baseline_metrics_path.exists()

print("Saved:", baseline_metrics_path)

print("\nSUCCESS: M10.1 baseline evaluation saved!")

Saved: C:\Users\fatim\Documents\network-intrusion-eda\reports\ml\unsw_lr_baseline_metrics.csv

SUCCESS: M10.1 baseline evaluation saved!


## M10.2 — Random Forest Classification

### Objective

Train a Random Forest classifier using the same
development training and validation partitions
as Logistic Regression.

### Methodology

- Dataset: UNSW-NB15
- Task: Binary classification
- Model: Random Forest
- Preprocessing: Numerical imputation and categorical encoding
- Training: Development training partition
- Evaluation: Development validation partition
- Official test: Not used

### Evaluation

Compare Random Forest against Logistic Regression
using accuracy, precision, recall, F1-score,
average precision, and false-positive rate.

In [31]:
from sklearn.ensemble import RandomForestClassifier

print("SUCCESS: Random Forest imported!")

SUCCESS: Random Forest imported!


In [32]:
rf_pipeline = Pipeline(
    steps=[
        (
            "preprocessing",
            clone(preprocessor_rf)
        ),
        (
            "classifier",
            RandomForestClassifier(
                n_estimators=100,
                max_depth=20,
                min_samples_leaf=2,
                max_features="sqrt",
                n_jobs=2,
                random_state=42
            )
        )
    ]
)

print(rf_pipeline)

print("\nSUCCESS: Random Forest pipeline created!")

Pipeline(steps=[('preprocessing',
                 ColumnTransformer(transformers=[('numeric',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median'))]),
                                                  ['dur', 'spkts', 'dpkts',
                                                   'sbytes', 'dbytes', 'rate',
                                                   'sttl', 'dttl', 'sload',
                                                   'dload', 'sloss', 'dloss',
                                                   'sinpkt', 'dinpkt', 'sjit',
                                                   'djit', 'swin', 'stcpb',
                                                   'dtcpb', 'dwin', 'tcprtt',
                                                   'synack', 'ackdat', 'smean',
                                                   'dmean', 'trans_depth',
                          

In [33]:
print("Starting Random Forest training...")

rf_start_time = time.time()

rf_pipeline.fit(
    X_dev_train,
    y_dev_train
)

rf_training_seconds = (
    time.time() - rf_start_time
)

print("\nTraining completed!")

print(
    "Training time:",
    round(rf_training_seconds, 2),
    "seconds"
)

print("\nSUCCESS: Random Forest trained!")

Starting Random Forest training...

Training completed!
Training time: 122.93 seconds

SUCCESS: Random Forest trained!


In [34]:
rf_val_predictions = rf_pipeline.predict(
    X_dev_val
)

rf_val_probabilities = (
    rf_pipeline.predict_proba(X_dev_val)[:, 1]
)

print("Validation records:", len(X_dev_val))

print("\nPredicted classes:")
print(
    pd.Series(rf_val_predictions).value_counts()
)

print("\nSUCCESS: Random Forest predictions generated!")

Validation records: 35196

Predicted classes:
1    24781
0    10415
Name: count, dtype: int64

SUCCESS: Random Forest predictions generated!


In [35]:
rf_metrics = {
    "Model": "Random Forest",
    "Accuracy": accuracy_score(
        y_dev_val,
        rf_val_predictions
    ),
    "Precision": precision_score(
        y_dev_val,
        rf_val_predictions,
        zero_division=0
    ),
    "Recall": recall_score(
        y_dev_val,
        rf_val_predictions,
        zero_division=0
    ),
    "F1": f1_score(
        y_dev_val,
        rf_val_predictions,
        zero_division=0
    ),
    "Average Precision": average_precision_score(
        y_dev_val,
        rf_val_probabilities
    ),
    "Training Seconds": rf_training_seconds
}

rf_metrics_df = pd.DataFrame([rf_metrics])

display(rf_metrics_df.round(4))

print("\nSUCCESS: Random Forest metrics calculated!")

,Model,Accuracy,Precision,Recall,F1,Average Precision,Training Seconds
0,Random Forest,0.9536,0.9528,0.9808,0.9666,0.9966,122.9254



SUCCESS: Random Forest metrics calculated!


In [36]:
rf_cm = confusion_matrix(
    y_dev_val,
    rf_val_predictions,
    labels=[0, 1]
)

rf_cm_df = pd.DataFrame(
    rf_cm,
    index=["Actual Normal", "Actual Attack"],
    columns=["Predicted Normal", "Predicted Attack"]
)

display(rf_cm_df)

print("\nClassification Report:")

print(
    classification_report(
        y_dev_val,
        rf_val_predictions,
        target_names=["Normal", "Attack"],
        zero_division=0
    )
)

,Predicted Normal,Predicted Attack
Actual Normal,9953,1170
Actual Attack,462,23611



Classification Report:
              precision    recall  f1-score   support

      Normal       0.96      0.89      0.92     11123
      Attack       0.95      0.98      0.97     24073

    accuracy                           0.95     35196
   macro avg       0.95      0.94      0.95     35196
weighted avg       0.95      0.95      0.95     35196



In [37]:
model_comparison_df = pd.concat(
    [
        lr_metrics_df,
        rf_metrics_df
    ],
    ignore_index=True
)

display(
    model_comparison_df.round(4)
)

print("\nSUCCESS: Model comparison generated!")

,Model,Accuracy,Precision,Recall,F1,Average Precision,Training Seconds
0,Logistic Regression,0.9358,0.9244,0.9870,0.9546,0.9920,17.8637
1,Random Forest,0.9536,0.9528,0.9808,0.9666,0.9966,122.9254



SUCCESS: Model comparison generated!


In [38]:
comparison_path = (
    ml_reports_dir / "unsw_model_comparison.csv"
)

model_comparison_df.to_csv(
    comparison_path,
    index=False
)

assert comparison_path.exists()

print("Saved:", comparison_path)

print("\nSUCCESS: M10.2 model comparison saved!")

Saved: C:\Users\fatim\Documents\network-intrusion-eda\reports\ml\unsw_model_comparison.csv

SUCCESS: M10.2 model comparison saved!


## M11.1 — Decision Threshold Analysis

### Objective

Investigate the trade-off between attack detection
and false-positive rates.

### Methodology

- Use development-validation predictions only.
- Evaluate a predefined set of thresholds.
- Compare Logistic Regression and Random Forest.
- Prioritize attack recall while controlling false alarms.
- Do not access the official test partition.

### Model Selection

Select a model and threshold using validation results
before performing final test evaluation.

In [39]:
import numpy as np
import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

# Predefined thresholds
thresholds = np.arange(0.20, 0.81, 0.05)

results = []

models = {
    "Logistic Regression": lr_val_probabilities,
    "Random Forest": rf_val_probabilities
}

for model_name, probabilities in models.items():

    for threshold in thresholds:

        predictions = (
            probabilities >= threshold
        ).astype(int)

        tn, fp, fn, tp = confusion_matrix(
            y_dev_val,
            predictions,
            labels=[0, 1]
        ).ravel()

        results.append({
            "Model": model_name,
            "Threshold": round(float(threshold), 2),
            "Accuracy": accuracy_score(
                y_dev_val, predictions
            ),
            "Precision": precision_score(
                y_dev_val, predictions,
                zero_division=0
            ),
            "Recall": recall_score(
                y_dev_val, predictions,
                zero_division=0
            ),
            "F1": f1_score(
                y_dev_val, predictions,
                zero_division=0
            ),
            "False Positive Rate": fp / (fp + tn),
            "False Positives": int(fp),
            "False Negatives": int(fn)
        })

threshold_df = pd.DataFrame(results)

display(threshold_df.round(4))

print("SUCCESS: Threshold analysis completed!")

,Model,Threshold,Accuracy,Precision,Recall,F1,False Positive Rate,False Positives,False Negatives
0,Logistic Regression,0.20,0.9341,0.9148,0.9965,0.9539,0.2008,2234,85
1,Logistic Regression,0.25,0.9353,0.9173,0.9951,0.9547,0.1941,2159,117
2,Logistic Regression,0.30,0.9353,0.9183,0.9939,0.9546,0.1915,2130,147
3,Logistic Regression,0.35,0.9349,0.9186,0.9927,0.9542,0.1903,2117,176
4,Logistic Regression,0.40,0.9352,0.9198,0.9918,0.9544,0.1871,2081,198
5,Logistic Regression,0.45,0.9360,0.9220,0.9902,0.9549,0.1814,2018,235
6,Logistic Regression,0.50,0.9358,0.9244,0.9870,0.9546,0.1748,1944,314
7,Logistic Regression,0.55,0.9364,0.9284,0.9828,0.9548,0.1641,1825,415
8,Logistic Regression,0.60,0.9368,0.9351,0.9752,0.9547,0.1465,1630,596
9,Logistic Regression,0.65,0.9341,0.9440,0.9607,0.9523,0.1234,1373,946


SUCCESS: Threshold analysis completed!


In [40]:
rf_threshold_df = threshold_df[
    threshold_df["Model"] == "Random Forest"
].copy()

display(
    rf_threshold_df[
        [
            "Threshold",
            "Precision",
            "Recall",
            "F1",
            "False Positive Rate",
            "False Positives",
            "False Negatives"
        ]
    ].round(4)
)

print("SUCCESS: Random Forest thresholds inspected!")

,Threshold,Precision,Recall,F1,False Positive Rate,False Positives,False Negatives
13,0.20,0.9160,0.9999,0.9561,0.1984,2207,3
14,0.25,0.9191,0.9995,0.9576,0.1903,2117,13
15,0.30,0.9231,0.9987,0.9594,0.1801,2003,32
16,0.35,0.9286,0.9965,0.9613,0.1659,1845,84
17,0.40,0.9356,0.9929,0.9634,0.1480,1646,170
18,0.45,0.9441,0.9885,0.9658,0.1266,1408,277
19,0.50,0.9528,0.9808,0.9666,0.1052,1170,462
20,0.55,0.9625,0.9698,0.9661,0.0818,910,727
21,0.60,0.9710,0.9573,0.9641,0.0619,688,1027
22,0.65,0.9767,0.9470,0.9616,0.0489,544,1277


SUCCESS: Random Forest thresholds inspected!


In [41]:
minimum_recall = 0.985

eligible_models = threshold_df[
    threshold_df["Recall"] >= minimum_recall
].copy()

if eligible_models.empty:

    print(
        "No evaluated configuration meets",
        "the minimum recall target."
    )

else:

    ranked_models = eligible_models.sort_values(
        by=[
            "False Positive Rate",
            "F1"
        ],
        ascending=[True, False]
    )

    print("Configurations meeting recall target:")

    display(
        ranked_models.head(10).round(4)
    )

    print("\nLeading candidate:")

    display(
        ranked_models.head(1).round(4)
    )

print("\nSUCCESS: Candidate configurations reviewed!")

Configurations meeting recall target:


,Model,Threshold,Accuracy,Precision,Recall,F1,False Positive Rate,False Positives,False Negatives
18,Random Forest,0.45,0.9521,0.9441,0.9885,0.9658,0.1266,1408,277
17,Random Forest,0.40,0.9484,0.9356,0.9929,0.9634,0.1480,1646,170
16,Random Forest,0.35,0.9452,0.9286,0.9965,0.9613,0.1659,1845,84
6,Logistic Regression,0.50,0.9358,0.9244,0.9870,0.9546,0.1748,1944,314
15,Random Forest,0.30,0.9422,0.9231,0.9987,0.9594,0.1801,2003,32
5,Logistic Regression,0.45,0.9360,0.9220,0.9902,0.9549,0.1814,2018,235
4,Logistic Regression,0.40,0.9352,0.9198,0.9918,0.9544,0.1871,2081,198
14,Random Forest,0.25,0.9395,0.9191,0.9995,0.9576,0.1903,2117,13
3,Logistic Regression,0.35,0.9349,0.9186,0.9927,0.9542,0.1903,2117,176
2,Logistic Regression,0.30,0.9353,0.9183,0.9939,0.9546,0.1915,2130,147



Leading candidate:


,Model,Threshold,Accuracy,Precision,Recall,F1,False Positive Rate,False Positives,False Negatives
18,Random Forest,0.45,0.9521,0.9441,0.9885,0.9658,0.1266,1408,277



SUCCESS: Candidate configurations reviewed!


In [42]:
threshold_path = (
    ml_reports_dir / "unsw_threshold_analysis.csv"
)

threshold_df.to_csv(
    threshold_path,
    index=False
)

assert threshold_path.exists()

print("Saved:", threshold_path)

print("SUCCESS: M11.1 threshold analysis saved!")

Saved: C:\Users\fatim\Documents\network-intrusion-eda\reports\ml\unsw_threshold_analysis.csv
SUCCESS: M11.1 threshold analysis saved!


## M11.2 — Final Model Selection

### Selected Configuration

- Model: Random Forest
- Decision threshold: 0.45
- Selection dataset: Development validation
- Minimum required attack recall: 98.5%
- Selection criterion: Lowest false-positive rate among eligible configurations

### Rationale

Random Forest at threshold 0.45 achieved 98.85% attack
recall and a 12.66% false-positive rate.

It reduced both false negatives and false positives
compared with Logistic Regression at threshold 0.50.

### Evaluation Protocol

The selected model configuration and decision threshold
are locked before final test evaluation.

The official test partition must not be used to modify
the selected configuration.

In [43]:
selected_model_name = "Random Forest"
selected_threshold = 0.45

selected_config = {
    "Model": selected_model_name,
    "Threshold": selected_threshold,
    "Minimum Recall Requirement": 0.985,
    "Selection Criterion": (
        "Lowest validation false-positive rate "
        "subject to recall >= 0.985"
    ),
    "Selection Dataset": "Development Validation",
    "Random State": 42
}

selected_config_df = pd.DataFrame(
    [selected_config]
)

display(selected_config_df)

selection_path = (
    ml_reports_dir / "unsw_selected_model_config.csv"
)

selected_config_df.to_csv(
    selection_path,
    index=False
)

assert selection_path.exists()

print("SUCCESS: M11.2 model selection locked!")

,Model,Threshold,Minimum Recall Requirement,Selection Criterion,Selection Dataset,Random State
0,Random Forest,0.45,0.985,Lowest validation false-positive rate subject ...,Development Validation,42


SUCCESS: M11.2 model selection locked!


In [44]:
required_variables = [
    "X_dev_train",
    "X_dev_val",
    "y_dev_train",
    "y_dev_val",
    "lr_pipeline",
    "rf_pipeline",
    "lr_val_probabilities",
    "rf_val_probabilities",
    "threshold_df",
    "preprocessor_rf",
    "unsw_train",
    "unsw_test",
]

missing_variables = [
    name
    for name in required_variables
    if name not in globals()
]

if missing_variables:
    print("Missing variables:")
    for name in missing_variables:
        print("-", name)
else:
    print("SUCCESS: ML environment restored!")

print("\nTraining records:", len(X_dev_train))
print("Validation records:", len(X_dev_val))

Missing variables:
- unsw_train
- unsw_test

Training records: 140145
Validation records: 35196


In [45]:
import pandas as pd

print("Available DataFrames:\n")

for name, value in globals().copy().items():
    if isinstance(value, pd.DataFrame):
        print(
            f"{name}: {value.shape}"
        )

Available DataFrames:

train_df: (175341, 45)
test_df: (82332, 45)
schema_df: (45, 4)
X_train_full: (175341, 42)
X_test_final: (82332, 42)
train_unique: (101040, 42)
test_unique: (53946, 42)
train_hash_label_df: (175341, 2)
test_hash_label_df: (82332, 2)
test_overlap_labels: (8541, 3)
overlap_audit: (8, 2)
train_candidates: (10279, 43)
test_candidates: (8541, 43)
train_candidate_unique: (1302, 43)
test_candidate_unique: (1302, 43)
exact_matches: (1302, 44)
X_dev_train: (140145, 42)
X_dev_val: (35196, 42)
split_assignments: (175341, 2)
lr_metrics_df: (1, 7)
lr_cm_df: (2, 2)
rf_metrics_df: (1, 7)
rf_cm_df: (2, 2)
model_comparison_df: (2, 7)
threshold_df: (26, 9)
rf_threshold_df: (13, 9)
eligible_models: (13, 9)
ranked_models: (13, 9)
selected_config_df: (1, 6)


In [46]:
# M11.3 — Final training input verification

print("FINAL MODEL TRAINING VERIFICATION")
print("-" * 45)

print("Training features:", X_train_full.shape)
print("Testing features:", X_test_final.shape)

print("\nTraining target distribution:")
print(train_df["label"].value_counts())

print("\nTesting target distribution:")
print(test_df["label"].value_counts())

print("\nSelected configuration:")
display(selected_config_df)

# Validate dataset dimensions
assert X_train_full.shape == (175341, 42)
assert X_test_final.shape == (82332, 42)

# Verify the target is not included in model inputs
assert "label" not in X_train_full.columns
assert "attack_cat" not in X_train_full.columns
assert "id" not in X_train_full.columns

# Ensure training and testing feature columns match
assert X_train_full.columns.equals(
    X_test_final.columns
)

# Verify the selected model and threshold
assert selected_config_df.loc[0, "Model"] == "Random Forest"
assert float(selected_config_df.loc[0, "Threshold"]) == 0.45

print("\nSUCCESS: Final training inputs verified!")

FINAL MODEL TRAINING VERIFICATION
---------------------------------------------
Training features: (175341, 42)
Testing features: (82332, 42)

Training target distribution:
label
1    119341
0     56000
Name: count, dtype: int64

Testing target distribution:
label
1    45332
0    37000
Name: count, dtype: int64

Selected configuration:


,Model,Threshold,Minimum Recall Requirement,Selection Criterion,Selection Dataset,Random State
0,Random Forest,0.45,0.985,Lowest validation false-positive rate subject ...,Development Validation,42



SUCCESS: Final training inputs verified!


In [47]:
# M11.3 — Create final Random Forest pipeline

from sklearn.base import clone

final_rf_pipeline = clone(rf_pipeline)

print("Final Random Forest pipeline:")
print(final_rf_pipeline)

print("\nSUCCESS: Final Random Forest pipeline created!")

Final Random Forest pipeline:
Pipeline(steps=[('preprocessing',
                 ColumnTransformer(transformers=[('numeric',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median'))]),
                                                  ['dur', 'spkts', 'dpkts',
                                                   'sbytes', 'dbytes', 'rate',
                                                   'sttl', 'dttl', 'sload',
                                                   'dload', 'sloss', 'dloss',
                                                   'sinpkt', 'dinpkt', 'sjit',
                                                   'djit', 'swin', 'stcpb',
                                                   'dtcpb', 'dwin', 'tcprtt',
                                                   'synack', 'ackdat', 'smean',
                                                   'dmean', 'trans_dept

In [48]:
# M11.3 — Prepare final training target

y_train_full = train_df["label"].astype(int)

print("Training features:", X_train_full.shape)
print("Training labels:", y_train_full.shape)

assert len(X_train_full) == len(y_train_full)

assert set(y_train_full.unique()) == {0, 1}

print("\nSUCCESS: Final training target prepared!")

Training features: (175341, 42)
Training labels: (175341,)

SUCCESS: Final training target prepared!


In [49]:
# M11.3 — Train final Random Forest model

import time

print("Starting final Random Forest training...")

final_training_start = time.time()

final_rf_pipeline.fit(
    X_train_full,
    y_train_full
)

final_training_seconds = (
    time.time() - final_training_start
)

print("\nFinal training completed!")

print(
    "Training time:",
    round(final_training_seconds, 2),
    "seconds"
)

print("\nSUCCESS: Final Random Forest trained!")

Starting final Random Forest training...

Final training completed!
Training time: 159.9 seconds

SUCCESS: Final Random Forest trained!


In [50]:
# M11.4 — Prepare official test labels

y_test_final = test_df["label"].astype(int)

print("Official test features:", X_test_final.shape)
print("Official test labels:", y_test_final.shape)

assert len(X_test_final) == len(y_test_final)

assert set(y_test_final.unique()) == {0, 1}

print("\nSUCCESS: Official test labels prepared!")

Official test features: (82332, 42)
Official test labels: (82332,)

SUCCESS: Official test labels prepared!


In [51]:
# M11.4 — Generate official test predictions

import numpy as np

final_test_probabilities = (
    final_rf_pipeline.predict_proba(
        X_test_final
    )[:, 1]
)

final_test_predictions = (
    final_test_probabilities >= 0.45
).astype(int)

print("Total predictions:", len(final_test_predictions))

print("\nPredicted class distribution:")
print(
    pd.Series(
        final_test_predictions,
        name="Predicted Label"
    ).value_counts()
)

assert len(final_test_predictions) == 82332

print("\nSUCCESS: Official test predictions generated!")

Total predictions: 82332

Predicted class distribution:
Predicted Label
1    57317
0    25015
Name: count, dtype: int64

SUCCESS: Official test predictions generated!


In [52]:
# M11.4 — Calculate official test performance

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    average_precision_score,
    confusion_matrix,
    classification_report
)

final_test_cm = confusion_matrix(
    y_test_final,
    final_test_predictions,
    labels=[0, 1]
)

tn, fp, fn, tp = final_test_cm.ravel()

final_test_metrics = {
    "Model": "Random Forest",
    "Dataset": "UNSW-NB15 Official Test",
    "Threshold": 0.45,
    "Records": len(y_test_final),
    "Accuracy": accuracy_score(
        y_test_final,
        final_test_predictions
    ),
    "Precision": precision_score(
        y_test_final,
        final_test_predictions,
        zero_division=0
    ),
    "Recall": recall_score(
        y_test_final,
        final_test_predictions,
        zero_division=0
    ),
    "F1": f1_score(
        y_test_final,
        final_test_predictions,
        zero_division=0
    ),
    "Average Precision": average_precision_score(
        y_test_final,
        final_test_probabilities
    ),
    "False Positive Rate": fp / (fp + tn),
    "False Positives": int(fp),
    "False Negatives": int(fn)
}

final_test_metrics_df = pd.DataFrame(
    [final_test_metrics]
)

display(final_test_metrics_df.round(4))

print("\nSUCCESS: Official test metrics calculated!")

,Model,Dataset,Threshold,Records,Accuracy,Precision,Recall,F1,Average Precision,False Positive Rate,False Positives,False Negatives
0,Random Forest,UNSW-NB15 Official Test,0.45,82332,0.8479,0.7862,0.994,0.878,0.9874,0.3312,12255,270



SUCCESS: Official test metrics calculated!


In [53]:
# M11.4 — Official test confusion matrix

final_test_cm_df = pd.DataFrame(
    final_test_cm,
    index=[
        "Actual Normal",
        "Actual Attack"
    ],
    columns=[
        "Predicted Normal",
        "Predicted Attack"
    ]
)

display(final_test_cm_df)

print("\nClassification Report:")

print(
    classification_report(
        y_test_final,
        final_test_predictions,
        target_names=[
            "Normal",
            "Attack"
        ],
        zero_division=0
    )
)

print("\nSUCCESS: Official test confusion matrix generated!")

,Predicted Normal,Predicted Attack
Actual Normal,24745,12255
Actual Attack,270,45062



Classification Report:
              precision    recall  f1-score   support

      Normal       0.99      0.67      0.80     37000
      Attack       0.79      0.99      0.88     45332

    accuracy                           0.85     82332
   macro avg       0.89      0.83      0.84     82332
weighted avg       0.88      0.85      0.84     82332


SUCCESS: Official test confusion matrix generated!


In [54]:
# M11.4 — Save official test evaluation

final_test_metrics_path = (
    ml_reports_dir / "unsw_final_test_metrics.csv"
)

final_test_metrics_df.to_csv(
    final_test_metrics_path,
    index=False
)

final_test_cm_path = (
    ml_reports_dir / "unsw_final_test_confusion_matrix.csv"
)

final_test_cm_df.to_csv(
    final_test_cm_path
)

assert final_test_metrics_path.exists()
assert final_test_cm_path.exists()

print("Saved:", final_test_metrics_path)
print("Saved:", final_test_cm_path)

print("\nSUCCESS: M11.4 official test evaluation saved!")

Saved: C:\Users\fatim\Documents\network-intrusion-eda\reports\ml\unsw_final_test_metrics.csv
Saved: C:\Users\fatim\Documents\network-intrusion-eda\reports\ml\unsw_final_test_confusion_matrix.csv

SUCCESS: M11.4 official test evaluation saved!


## M11.5 — Exact Train–Test Feature Overlap Analysis

### Objective

Evaluate the final Random Forest on official test records
that do not have an identical feature vector in the
official training dataset.

### Methodology

- Compare all 42 input features.
- Identify exact feature-vector matches.
- Exclude overlapping test records.
- Evaluate the previously selected Random Forest.
- Keep the decision threshold fixed at 0.45.

### Research Purpose

Investigate whether the official test performance differs
between the complete test dataset and the subset without
exact training-feature overlap.

This analysis does not establish performance on live traffic.

In [55]:
# M11.5 — Identify exact train-test feature overlap

import pandas as pd
import numpy as np

# Combine feature vectors and remove duplicates
unique_train_features = X_train_full.drop_duplicates()

# Merge test features with unique training features
# to identify exact matches across all 42 columns.
test_overlap_check = X_test_final.merge(
    unique_train_features,
    how="left",
    on=list(X_train_full.columns),
    indicator=True,
    sort=False
)

# Merge preserves the test row order because the right-hand
# feature combinations are unique.
assert len(test_overlap_check) == len(X_test_final)

exact_overlap_mask = (
    test_overlap_check["_merge"]
    .eq("both")
    .to_numpy()
)

non_overlap_mask = ~exact_overlap_mask

print(
    "Total official test records:",
    len(X_test_final)
)

print(
    "Exact overlapping test records:",
    int(exact_overlap_mask.sum())
)

print(
    "Non-overlapping test records:",
    int(non_overlap_mask.sum())
)

assert len(non_overlap_mask) == len(X_test_final)

assert non_overlap_mask.sum() > 0

print("\nSUCCESS: Exact overlap mask created!")

Total official test records: 82332
Exact overlapping test records: 8541
Non-overlapping test records: 73791

SUCCESS: Exact overlap mask created!


In [56]:
# M11.5 — Evaluate non-overlapping test records

y_test_non_overlap = y_test_final.to_numpy()[
    non_overlap_mask
]

predictions_non_overlap = final_test_predictions[
    non_overlap_mask
]

probabilities_non_overlap = final_test_probabilities[
    non_overlap_mask
]

non_overlap_cm = confusion_matrix(
    y_test_non_overlap,
    predictions_non_overlap,
    labels=[0, 1]
)

tn, fp, fn, tp = non_overlap_cm.ravel()

non_overlap_metrics = {
    "Dataset": "UNSW-NB15 Non-Overlapping Test",
    "Records": len(y_test_non_overlap),
    "Threshold": 0.45,
    "Accuracy": accuracy_score(
        y_test_non_overlap,
        predictions_non_overlap
    ),
    "Precision": precision_score(
        y_test_non_overlap,
        predictions_non_overlap,
        zero_division=0
    ),
    "Recall": recall_score(
        y_test_non_overlap,
        predictions_non_overlap,
        zero_division=0
    ),
    "F1": f1_score(
        y_test_non_overlap,
        predictions_non_overlap,
        zero_division=0
    ),
    "Average Precision": average_precision_score(
        y_test_non_overlap,
        probabilities_non_overlap
    ),
    "False Positive Rate": fp / (fp + tn),
    "False Positives": int(fp),
    "False Negatives": int(fn)
}

non_overlap_metrics_df = pd.DataFrame(
    [non_overlap_metrics]
)

display(non_overlap_metrics_df.round(4))

print("\nNon-overlapping confusion matrix:")

display(
    pd.DataFrame(
        non_overlap_cm,
        index=["Actual Normal", "Actual Attack"],
        columns=["Predicted Normal", "Predicted Attack"]
    )
)

print(
    "\nSUCCESS: Non-overlapping test evaluation completed!"
)

,Dataset,Records,Threshold,Accuracy,Precision,Recall,F1,Average Precision,False Positive Rate,False Positives,False Negatives
0,UNSW-NB15 Non-Overlapping Test,73791,0.45,0.8375,0.7641,0.9932,0.8637,0.9835,0.3302,11731,259



Non-overlapping confusion matrix:


,Predicted Normal,Predicted Attack
Actual Normal,23797,11731
Actual Attack,259,38004



SUCCESS: Non-overlapping test evaluation completed!


In [57]:
# M11.5 — Save non-overlapping evaluation

non_overlap_path = (
    ml_reports_dir / "unsw_non_overlap_test_metrics.csv"
)

non_overlap_metrics_df.to_csv(
    non_overlap_path,
    index=False
)

assert non_overlap_path.exists()

print("Saved:", non_overlap_path)

print("\nSUCCESS: M11.5 overlap analysis saved!")

Saved: C:\Users\fatim\Documents\network-intrusion-eda\reports\ml\unsw_non_overlap_test_metrics.csv

SUCCESS: M11.5 overlap analysis saved!


In [58]:
import joblib

print("SUCCESS: Joblib imported!")

SUCCESS: Joblib imported!


In [59]:
from pathlib import Path

models_dir = project_dir / "models"

models_dir.mkdir(
    parents=True,
    exist_ok=True
)

print("Model directory:", models_dir)

print("\nSUCCESS: Model directory ready!")

Model directory: C:\Users\fatim\Documents\network-intrusion-eda\models

SUCCESS: Model directory ready!


In [60]:
# M12.1 — Save final Random Forest pipeline

model_path = (
    models_dir / "unsw_random_forest_pipeline.joblib"
)

joblib.dump(
    final_rf_pipeline,
    model_path,
    compress=3
)

assert model_path.exists()

model_size_mb = (
    model_path.stat().st_size / (1024 ** 2)
)

print("Saved model:", model_path)

print(
    "Model size:",
    round(model_size_mb, 2),
    "MB"
)

print("\nSUCCESS: Final ML pipeline saved!")

Saved model: C:\Users\fatim\Documents\network-intrusion-eda\models\unsw_random_forest_pipeline.joblib
Model size: 7.35 MB

SUCCESS: Final ML pipeline saved!


In [61]:
# M12.1 — Save model metadata

import json

model_metadata = {
    "model_name": "Random Forest",
    "dataset": "UNSW-NB15",
    "task": "Binary Network Intrusion Classification",
    "threshold": 0.45,
    "normal_label": 0,
    "attack_label": 1,
    "feature_count": len(X_train_full.columns),
    "feature_columns": list(X_train_full.columns),
    "training_records": len(X_train_full),
    "official_test_records": len(X_test_final),
    "training_seconds": round(
        final_training_seconds,
        2
    ),
    "limitations": [
        "Model evaluated on benchmark network traffic.",
        "High false-positive rate on official test data.",
        "Not validated for real-time production deployment."
    ]
}

metadata_path = (
    models_dir / "unsw_model_metadata.json"
)

with open(
    metadata_path,
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        model_metadata,
        file,
        indent=4
    )

assert metadata_path.exists()

print("Saved metadata:", metadata_path)

print("\nSUCCESS: Model metadata saved!")

Saved metadata: C:\Users\fatim\Documents\network-intrusion-eda\models\unsw_model_metadata.json

SUCCESS: Model metadata saved!


In [62]:
# M12.1 — Verify saved model

loaded_pipeline = joblib.load(
    model_path
)

sample_features = X_test_final.iloc[:10].copy()

original_probabilities = (
    final_rf_pipeline.predict_proba(
        sample_features
    )[:, 1]
)

loaded_probabilities = (
    loaded_pipeline.predict_proba(
        sample_features
    )[:, 1]
)

np.testing.assert_allclose(
    original_probabilities,
    loaded_probabilities,
    rtol=0,
    atol=1e-12
)

with open(
    metadata_path,
    "r",
    encoding="utf-8"
) as file:
    loaded_metadata = json.load(file)

assert loaded_metadata["threshold"] == 0.45

assert loaded_metadata["feature_count"] == 42

print("Original probabilities:")
print(original_probabilities)

print("\nLoaded probabilities:")
print(loaded_probabilities)

print("\nSUCCESS: Saved model verified!")

Original probabilities:
[0.84384168 0.86053325 0.80211324 0.76994513 0.7668139  0.83883788
 0.79642239 0.80124905 0.00514131 0.00514131]

Loaded probabilities:
[0.84384168 0.86053325 0.80211324 0.76994513 0.7668139  0.83883788
 0.79642239 0.80124905 0.00514131 0.00514131]

SUCCESS: Saved model verified!
